# 07 - Git + dbt: transformações como receitas versionadas

**Objetivo:** rodar um projeto **dbt** que está no GitHub, direto no Snowflake.

**Analogia:**
- **Git** é o histórico de versões do Google Docs, só que para código: quem mudou, o quê e quando.
- **dbt** é um livro de receitas. Cada modelo é uma receita SQL, com teste de qualidade e documentação.

```mermaid
flowchart LR
  DEV[Analista edita modelo] -->|git push| GH[GitHub]
  GH -->|FETCH| REPO[HOL_REPO]
  REPO -->|CREATE DBT PROJECT| DBT[Projeto dbt no Snowflake]
  DBT -->|EXECUTE build| T[Tabelas GOLD_DBT + testes]
```

**Tempo estimado:** 15 minutos

In [ ]:
%%sql -r dataframe_1
-- Contexto da sessao: quem sou eu, qual motor e qual gaveta vou usar.
-- Rode esta celula SEMPRE que abrir o notebook.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE ESPM_WH;
USE DATABASE ESPM_HOL;
USE SCHEMA GOLD_DBT;

## Passo 1 - O que tem no projeto dbt?

In [ ]:
%%sql -r dataframe_2
ALTER GIT REPOSITORY ESPM_HOL.PUBLIC.HOL_REPO FETCH;
LS @ESPM_HOL.PUBLIC.HOL_REPO/branches/main/dbt_project/;

| Arquivo | Para que serve |
|---|---|
| `dbt_project.yml` | nome do projeto e configurações gerais |
| `profiles.yml` | onde gravar (database, schema, warehouse) |
| `models/sources.yml` | de onde vêm os dados (a nossa Silver) |
| `models/*.sql` | as receitas |
| `models/schema.yml` | testes de qualidade (ex.: coluna não pode ser nula) |

## Passo 2 - Criar o projeto dbt a partir do Git

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE DBT PROJECT ESPM_HOL.GOLD_DBT.LOJA_PAULISTA_DBT
  FROM '@ESPM_HOL.PUBLIC.HOL_REPO/branches/main/dbt_project'
  COMMENT = 'Projeto dbt do HOL ESPM';

SHOW DBT PROJECTS IN SCHEMA ESPM_HOL.GOLD_DBT;

## Passo 3 - Executar: `build` = rodar modelos **e** testes

In [ ]:
%%sql -r dataframe_4
EXECUTE DBT PROJECT ESPM_HOL.GOLD_DBT.LOJA_PAULISTA_DBT ARGS = 'build';

**O que observar:** a coluna `SUCCESS` deve ser `TRUE`. Em `STDOUT` aparecem os modelos criados e os testes com `PASS`.

In [ ]:
%%sql -r dataframe_5
-- Resultado do modelo receita_por_categoria
SELECT * FROM ESPM_HOL.GOLD_DBT.RECEITA_POR_CATEGORIA ORDER BY receita DESC;

### EXERCICIO 1 - Rode apenas UM modelo do dbt (o de clientes por cidade)

Complete os `???` na célula abaixo.

**Dica:** o nome do modelo é o nome do arquivo .sql, sem a extensão. Veja a lista do Passo 1.

**Travou?** Abra o Cortex Code e peça:
> No dbt, como rodo só o modelo clientes_por_cidade?

In [ ]:
%%sql -r dataframe_6
EXECUTE DBT PROJECT ESPM_HOL.GOLD_DBT.LOJA_PAULISTA_DBT
  ARGS = 'run --select ???';

In [ ]:
%%sql -r dataframe_7
SELECT * FROM ESPM_HOL.GOLD_DBT.CLIENTES_POR_CIDADE ORDER BY clientes DESC;

## Desafio extra (opcional): editar uma receita

1. No Workspace, abra `dbt_project/models/clientes_por_cidade.sql`.
2. Peça ao Cortex Code: *"adicione uma coluna com o ticket médio por cidade"*.
3. Use o botão **Deploy** do dbt no Workspace para criar sua própria versão do projeto e execute.

No mundo real, essa mudança iria para o GitHub via **pull request**, alguém revisaria e só então ela entraria em produção.

| | Dynamic Tables | Stream + Task | dbt |
|---|---|---|---|
| Ideal para | pipeline contínuo e declarativo | reagir a eventos | times com ciclo de desenvolvimento |
| Esforço | baixo | médio | médio |
| Testes de qualidade | via DMF | manual | nativo (`schema.yml`) |

**Próximo:** `08_consumo_streamlit_marketplace`.